# Task 3: Federal Reserve Estimate Replication and Feature Preprocessing for 2025


This notebook uses the 2025 Survey of Household Economics and Decisionmaking (SHED) data to reproduce the Federal Reserve’s published emergency-expense estimate. It also helps prepare the categorical and numerical variables for future machine-learning models that we will create later on (October).


In [19]:
from google.colab import files
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler

In [4]:
uploaded = files.upload()

Saving public2025_clean.csv to public2025_clean (1).csv


In [5]:
df = pd.read_csv(
    "/content/public2025_clean.csv",
    low_memory=False
)

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (12934, 809)


,shedid,weight,weight_pop,panel_weight,panel_weight_pop,L0_a,L0_b,L0_c,L0_d,L0_e,...,E12_e_iflag,E12_f_iflag,E12_g_iflag,CH2A_iflag,race_5cat,inc_4cat_50k,educ_4cat,pay_casheqv,atleast_okay,malefemale
0,202304484,0.6467,13225.3400,NaN,NaN,Yes,No,No,No,No,...,0,0,0,0,White,"$50,000–$99,999",Some college/technical or associates degree,Yes,Yes,Male
1,202204577,0.9687,19810.3406,0.9357,56011.816,No,No,No,No,No,...,0,0,0,0,White,"$100,000 or more",Bachelor's degree or more,Yes,Yes,Female
2,202301342,1.0129,20715.6590,NaN,NaN,No,No,Yes,No,No,...,0,0,0,0,White,"$50,000–$99,999",Less than a high school degree,No,No,Female
3,202500830,1.0278,21020.6151,NaN,NaN,Yes,No,No,No,No,...,0,0,0,0,White,"$50,000–$99,999",High school degree or GED,Yes,Yes,Male
4,202504363,0.7688,15722.4241,NaN,NaN,Yes,No,No,No,No,...,0,0,0,0,White,"$50,000–$99,999",Some college/technical or associates degree,Yes,Yes,Female


##Data Validation

The revised dataset contains 12,934 survey respondents and has 809 variables. Before starting the analysis, the data is being checked for required columns, duplicate respondent IDs, missing population weights, and missing target responses. These checks confirm that all the necessary information is available and that every respondent has a unique identifier to differentiate them from one another.


In [6]:
required_columns = [
    "shedid",
    "weight_pop",
    "pay_casheqv"
]

missing_columns = [
    column for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

print("Data validation complete and all required columns are present.")
print("Number of duplicate respondent IDs:", df["shedid"].duplicated().sum())
print("Number of missing population weights:", df["weight_pop"].isna().sum())
print("Number of missing target values:", df["pay_casheqv"].isna().sum())

print("\nTarget response counts:")
print(df["pay_casheqv"].value_counts(dropna=False))

Data validation complete and all required columns are present.
Number of duplicate respondent IDs: 0
Number of missing population weights: 0
Number of missing target values: 0

Target response counts:
pay_casheqv
Yes    8398
No     4536
Name: count, dtype: int64


##Replication Standard

The Federal Reserve reports found that 63% of adults could cover a $400 emergency expense using cash or  equivalent. The replication is considered a success if the weighted estimate is within 0.5 percentage points of the published result. The `weight_pop' variable is used so the survey sample represents the U.S. adult population.



In [7]:
fed_estimate = 63.0
tolerance = 0.5

can_cover_expense = (
    df["pay_casheqv"]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("yes")
    .astype(int)
)

weighted_result = (
    (
        can_cover_expense * df["weight_pop"]
    ).sum()
    / df["weight_pop"].sum()
) * 100

regular_result = can_cover_expense.mean() * 100

estimate_gap = abs(
    weighted_result - fed_estimate
)

passed_check = estimate_gap <= tolerance

print(f"Federal Reserve result: {fed_estimate:.2f}%")
print(f"Our weighted result: {weighted_result:.2f}%")
print(f"Our result without weights(regular result): {regular_result:.2f}%")
print(f"Distance from published result: {estimate_gap:.2f} points")
print(f"Maximum allowed distance: {tolerance:.2f} points")
print(f"Result is within tolerance: {passed_check}")

Federal Reserve result: 63.00%
Our weighted result: 63.14%
Our result without weights(regular result): 64.93%
Distance from published result: 0.14 points
Maximum allowed distance: 0.50 points
Result is within tolerance: True


##Replication Result

Using the `weight_pop` survey weights, the estimated percentage of adults in the U.S. who could cover a $400 emergency expense using cash or the equivalent was 63.14%. The 63.14% is 0.14 percentage points from the Federal Reserve’s published estimate of 63%, so the replication passes the declared tolerance, which was 0.50 points.

The unweighted estimate was found to be 64.93%, which was more optimistic than the weighted estimate. This demonstrates why survey weights are necessary when using the SHED data to describe the U.S. adult population.


In [8]:
financial_fragility = df["pay_casheqv"].map({
    "No": 1,
    "Yes": 0
})

if financial_fragility.isna().any():
    raise ValueError(
        "The target contains a missing or unexpected response."
    )

population_weights = df["weight_pop"].copy()

print("Financial-fragility target counts:")
print(financial_fragility.value_counts().sort_index())

print("\nNumber of population weights:", len(population_weights))
print("Number of missing population weights:", population_weights.isna().sum())

Financial-fragility target counts:
pay_casheqv
0    8398
1    4536
Name: count, dtype: int64

Number of population weights: 12934
Number of missing population weights: 0


##Feature Selection

The original `pay_casheqv` variable measures whether a respondent could cover the emergency expense. This was then converted into a financial-fragility target, where 1 represents a respondent who could not cover the expense and 0 represents a respondent who could cover the emergency expense.

The survey weights are preserved separately for later weighted analysis. Respondent identifiers, survey weights, imputation flags, the original target, and the EF3 emergency-expense variables are excluded from the feature matrix. After the exclusions listed previously, the feature matrix contains 438 predictors.

In [9]:
imputation_flag_columns = [
    name for name in df.columns
    if name.endswith("_iflag")
]

emergency_answer_columns = [
    "EF3_a",
    "EF3_b",
    "EF3_c",
    "EF3_d",
    "EF3_e",
    "EF3_f",
    "EF3_g",
    "EF3_h"
]

non_feature_columns = [
    "shedid",
    "duration",
    "weight",
    "weight_pop",
    "panel_weight",
    "panel_weight_pop",
    "pay_casheqv"
]

excluded_columns = (
    imputation_flag_columns
    + emergency_answer_columns
    + non_feature_columns
)

print("Number of _iflag columns excluded:", len(imputation_flag_columns))
print("Number of EF3 columns excluded:", len(emergency_answer_columns))
print("Other non-feature columns excluded:", len(non_feature_columns))
print("Total columns excluded:", len(excluded_columns))

Number of _iflag columns excluded: 357
Number of EF3 columns excluded: 8
Other non-feature columns excluded: 7
Total columns excluded: 372


In [10]:
features = df.drop(
    columns=excluded_columns,
    errors="ignore"
).copy()

print("Original dataset shape:", df.shape)
print("Feature matrix shape:", features.shape)

Original dataset shape: (12934, 809)
Feature matrix shape: (12934, 438)


In [11]:
remaining_iflags = [
    name for name in features.columns
    if name.endswith("_iflag")
]

remaining_ef3_columns = [
    name for name in emergency_answer_columns
    if name in features.columns
]

print("Remaining _iflag variables:", len(remaining_iflags))
print("Remaining EF3 columns:", len(remaining_ef3_columns))
print("Target still in features:", "pay_casheqv" in features.columns)
print("Population weight still in features:", "weight_pop" in features.columns)

Remaining _iflag variables: 0
Remaining EF3 columns: 0
Target still in features: False
Population weight still in features: False


##Identify Feature Types
The feature matrix contains 415 categorical variables and 23 numerical variables. These groups are identified separately because they require different preprocessing methods.

Leading and trailing spaces are removed from categorical values, and empty strings are represented as missing values. The `__NOT_ASKED__` category is preserved because it represents survey skip logic rather than an ordinary missing response.


In [12]:
categorical_features = features.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

numerical_features = features.select_dtypes(
    include=[np.number]
).columns.tolist()

print("Categorical variables:", len(categorical_features))
print("Numerical variables:", len(numerical_features))
print("Total predictor variables:", len(features.columns))

Categorical variables: 415
Numerical variables: 23
Total predictor variables: 438


In [13]:
for column in categorical_features:
    features[column] = (
        features[column]
        .str.strip()
        .replace("", np.nan)
    )

print("Categorical values cleaned.")
print("The __NOT_ASKED__ category was preserved.")

Categorical values cleaned.
The __NOT_ASKED__ category was preserved.


##Train / Test
The data is divided into 80% being used for the training set, while the other 20% is used for the testing set. A stratified split is used so both sets maintain approximately the same percentages of financially fragile and non-fragile respondents.

A fixed random state of 42 makes the split reproducible. Preprocessing rules are learned only from the training data so information from the testing set does not influence model preparation.


In [17]:
(
    features_train,
    features_test,
    target_train,
    target_test,
    weights_train,
    weights_test
) = train_test_split(
    features,
    financial_fragility,
    population_weights,
    test_size=0.20,
    random_state=42,
    stratify=financial_fragility
)

print("Training rows and predictors:", features_train.shape)
print("Testing rows and predictors:", features_test.shape)

print("\nTraining outcome distribution:")
print("Able to cover expense (0):", (target_train == 0).sum())
print("Financially fragile (1):", (target_train == 1).sum())

print("\nTesting outcome distribution:")
print("Able to cover expense (0):", (target_test == 0).sum())
print("Financially fragile (1):", (target_test == 1).sum())

overlapping_rows = len(
    set(features_train.index).intersection(features_test.index)
)

print("\nRows appearing in both training and testing sets:", overlapping_rows)

Training rows and predictors: (10347, 438)
Testing rows and predictors: (2587, 438)

Training outcome distribution:
Able to cover expense (0): 6718
Financially fragile (1): 3629

Testing outcome distribution:
Able to cover expense (0): 1680
Financially fragile (1): 907

Rows appearing in both training and testing sets: 0


##Feature Preprocessing

Categorical and numerical features are processed separately. Genuine missing categorical values are represented by a separate `__MISSING__` category and the missing numerical values are replaced with the median that is calculated from the training data. The preprocessing pipeline is fitted only on the training data and remains unchanged from the testing data.


In [20]:
categorical_pipeline = Pipeline(
    steps=[
        (
            "fill_missing_categories",
            SimpleImputer(
                strategy="constant",
                fill_value="__MISSING__"
            )
        ),
        (
            "encode_categories",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

numerical_pipeline = Pipeline(
    steps=[
        (
            "fill_missing_numbers",
            SimpleImputer(
                strategy="median"
            )
        ),
        (
            "normalize_numbers",
            MinMaxScaler()
        )
    ]
)

In [22]:
feature_preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical_processing",
            categorical_pipeline,
            categorical_features
        ),
        (
            "numerical_processing",
            numerical_pipeline,
            numerical_features
        )
    ],
    remainder="drop"
)

print("Categorical predictors:", len(categorical_features))
print("Numerical predictors:", len(numerical_features))
print("Feature preprocessing pipeline has been created successfully.")

Categorical predictors: 415
Numerical predictors: 23
Feature preprocessing pipeline has been created successfully.


Apply Feature Preprocessing

In [24]:
prepared_training_features = (
    feature_preprocessor.fit_transform(features_train)
)

prepared_testing_features = (
    feature_preprocessor.transform(features_test)
)

prepared_feature_names = (
    feature_preprocessor.get_feature_names_out()
)

print(
    "Training shape after preprocessing:",
    prepared_training_features.shape
)

print(
    "Testing shape after preprocessing:",
    prepared_testing_features.shape
)

print(
    "Predictors available after preprocessing:",
    len(prepared_feature_names)
)

Training shape after preprocessing: (10347, 2376)
Testing shape after preprocessing: (2587, 2376)
Predictors available after preprocessing: 2376


In [25]:
missing_training_values = np.isnan(
    prepared_training_features.data
).sum()

missing_testing_values = np.isnan(
    prepared_testing_features.data
).sum()

print(
    "Missing values in the prepared training data:",
    missing_training_values
)

print(
    "Missing values in the prepared testing data:",
    missing_testing_values
)

print(
    "The training and testing columns match:",
    prepared_training_features.shape[1]
    == prepared_testing_features.shape[1]
)

Missing values in the prepared training data: 0
Missing values in the prepared testing data: 0
The training and testing columns match: True


Validation Checker

In [26]:
validation_checks = {
    "The training rows match target": (
        prepared_training_features.shape[0]
        == len(target_train)
    ),
    "The testing rows match target": (
        prepared_testing_features.shape[0]
        == len(target_test)
    ),
    "The training rows match weights": (
        prepared_training_features.shape[0]
        == len(weights_train)
    ),
    "The testing rows match weights": (
        prepared_testing_features.shape[0]
        == len(weights_test)
    ),
    "The training and testing columns match": (
        prepared_training_features.shape[1]
        == prepared_testing_features.shape[1]
    ),
    "The prepared feature names are unique": (
        len(prepared_feature_names)
        == len(set(prepared_feature_names))
    ),
    "The training data has no missing values": (
        np.isnan(prepared_training_features.data).sum() == 0
    ),
    "The testing data has no missing values": (
        np.isnan(prepared_testing_features.data).sum() == 0
    )
}

for check_name, passed in validation_checks.items():
    print(f"{check_name}: {passed}")

all_checks_passed = all(validation_checks.values())

print("\nAll preprocessing checks have sucessfully passed:", all_checks_passed)

The training rows match target: True
The testing rows match target: True
The training rows match weights: True
The testing rows match weights: True
The training and testing columns match: True
The prepared feature names are unique: True
The training data has no missing values: True
The testing data has no missing values: True

All preprocessing checks have sucessfully passed: True
